In [13]:
%cd /content/opg-live
!git fetch origin && git reset --hard origin/main
%cd /content/opg-live/scripts


/content/opg-live
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 711 bytes | 711.00 KiB/s, done.
From https://github.com/AndreasTopuh/opg-live
   f07bee6..7c49e31  main       -> origin/main
HEAD is now at 7c49e31 fix: decode masks per-image (SAM mask_decoder cannot batch images via repeat_interleave)
/content/opg-live/scripts


# 01 — Train Medical SAM Adapter (Phase A, Stage 2)

Train lesion-level adapter di DENTEX (Wu et al., 2025). SAM ViT-H frozen, hanya adapter + mask decoder dilatih (~2-3%). Checkpoint → Drive.

**Prasyarat:** `00_setup.ipynb` sudah sukses (Drive mount, SAM ViT-H di Drive, DENTEX verified).

Runtime → **T4/L4 GPU** → Run all. Training ~beberapa jam (ViT-H berat).

## Cell 1 — Mount Drive + clone/pull repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/opg-live'

import os
REPO = '/content/opg-live'
if not os.path.exists(REPO):
    !git clone https://github.com/AndreasTopuh/opg-live.git {REPO}
else:
    !cd {REPO} && git pull
%cd {REPO}/scripts

Mounted at /content/drive
Cloning into '/content/opg-live'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 49 (delta 21), reused 43 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 19.86 KiB | 19.86 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/opg-live/scripts


## Cell 2 — Install deps (kalau runtime baru)

In [3]:
%pip install -q segment-anything pycocotools opencv-python-headless
import torch; print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


## Cell 3 — Smoke test dataset (harus 3529 lesion, 1 sample valid)

In [ ]:
import os
os.environ['DRIVE_ROOT'] = DRIVE_ROOT
!python /content/opg-live/scripts/stage2/dentex_dataset.py

## Cell 4 — Cek trainable params (~2-3%) sebelum training
Sanity check adapter ke-inject benar & base frozen.

In [ ]:
import sys, torch
sys.path.insert(0, '/content/opg-live/scripts/stage2')
from segment_anything import sam_model_registry
from sam_adapter import inject_adapters, trainable_report

sam = sam_model_registry['vit_h'](checkpoint=f'{DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth')
inject_adapters(sam)
trainable_report(sam)   # target: a few % of 641M

## Cell 5 — Training
Mulai pendek dulu (`--epochs 2`) untuk pastikan loop jalan & loss turun. Kalau OK, naikkan epoch. Checkpoint terbaik otomatis ke `Drive/checkpoints/adapter_best.pth`.

In [ ]:
!python /content/opg-live/scripts/stage2/train_adapter.py \
  --drive {DRIVE_ROOT} \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 2 --bs 2 --accum 4 --lr 1e-4


Lesion: 3529 | train 3002 | val 527
Trainable: 56.59M / 694M  (8.16%)
  ep0 step0/1501 loss 0.1963
  ep0 step50/1501 loss 0.1395
  ep0 step100/1501 loss 0.1181
  ep0 step150/1501 loss 0.1079
  ep0 step200/1501 loss 0.1003
  ep0 step250/1501 loss 0.0948
  ep0 step300/1501 loss 0.0926
  ep0 step350/1501 loss 0.0904
  ep0 step400/1501 loss 0.0884
  ep0 step450/1501 loss 0.0876
  ep0 step500/1501 loss 0.0868
  ep0 step550/1501 loss 0.0857
  ep0 step600/1501 loss 0.0851
  ep0 step650/1501 loss 0.0841
  ep0 step700/1501 loss 0.0829
  ep0 step750/1501 loss 0.0822
  ep0 step800/1501 loss 0.0813
  ep0 step850/1501 loss 0.0805
  ep0 step900/1501 loss 0.0798
  ep0 step950/1501 loss 0.0793
  ep0 step1000/1501 loss 0.0789
  ep0 step1050/1501 loss 0.0789
  ep0 step1100/1501 loss 0.0786
  ep0 step1150/1501 loss 0.0779
  ep0 step1200/1501 loss 0.0775
  ep0 step1250/1501 loss 0.0771
  ep0 step1300/1501 loss 0.0769
  ep0 step1350/1501 loss 0.0768
  ep0 step1400/1501 loss 0.0765
  ep0 step1450/1501 loss 

## Cell 6 — Full training (setelah Cell 5 terbukti loss turun)
Naikkan epoch. **Checkpoint ke Drive = aman dari reset Colab.**

In [4]:
!python /content/opg-live/scripts/stage2/train_adapter.py \
  --drive {DRIVE_ROOT} \
  --sam_ckpt {DRIVE_ROOT}/checkpoints/sam_vit_h_4b8939.pth \
  --epochs 8 --bs 2 --accum 4 --lr 1e-4



Lesion: 3529 | train 3002 | val 527
Trainable: 56.59M / 694M  (8.16%)
  ep0 step0/1501 loss 0.2010
  ep0 step50/1501 loss 0.1396
  ep0 step100/1501 loss 0.1157
  ep0 step150/1501 loss 0.1060
  ep0 step200/1501 loss 0.1000
  ep0 step250/1501 loss 0.0969
  ep0 step300/1501 loss 0.0934
  ep0 step350/1501 loss 0.0911
  ep0 step400/1501 loss 0.0890
  ep0 step450/1501 loss 0.0869
  ep0 step500/1501 loss 0.0855
  ep0 step550/1501 loss 0.0841
  ep0 step600/1501 loss 0.0826
  ep0 step650/1501 loss 0.0819
  ep0 step700/1501 loss 0.0810
  ep0 step750/1501 loss 0.0805
  ep0 step800/1501 loss 0.0802
  ep0 step850/1501 loss 0.0796
  ep0 step900/1501 loss 0.0793
  ep0 step950/1501 loss 0.0788
  ep0 step1000/1501 loss 0.0782
  ep0 step1050/1501 loss 0.0779
  ep0 step1100/1501 loss 0.0773
  ep0 step1150/1501 loss 0.0770
  ep0 step1200/1501 loss 0.0765
  ep0 step1250/1501 loss 0.0761
  ep0 step1300/1501 loss 0.0758
  ep0 step1350/1501 loss 0.0755
  ep0 step1400/1501 loss 0.0753
  ep0 step1450/1501 loss 

## ✅ Deliverable Phase A (Stage 2)
- [ ] `adapter_best.pth` di Drive (~20-50 MB)
- [ ] val Dice global terlaporkan
- [ ] **Dice per-kelas** terlaporkan (Impacted/Caries/Periapical/Deep Caries) — penting untuk H4
- [ ] Target Dice ≥ 0.80 (lesion-level)

**Catatan H4:** Periapical n=158 → Dice-nya kemungkinan paling rendah & varian tinggi. Itu wajar, dilaporkan apa adanya (effect-size primary).

**Next:** `02_auto_pipeline.ipynb` (Stage 1 HierarchicalDet + Stage 2 adapter → 3-arm artifact: bbox / mask / hybrid).